In [1]:
import pandas as pd
import numpy as np

freight = pd.read_csv('freight_rates.csv', parse_dates=['date'])
ffa = pd.read_csv('ffa_prices.csv', parse_dates=['date'])
bunker = pd.read_csv('bunker_prices.csv', parse_dates=['date'])
coal_prices = pd.read_csv('coal_prices.csv', parse_dates=['date'])
port_calls = pd.read_csv('port_calls.csv', parse_dates=['arrival_time','berth_time','departure_time'])
ais_positions = pd.read_csv('ais_positions.csv', parse_dates=['timestamp'])
weather = pd.read_csv('weather.csv', parse_dates=['timestamp'])
events = pd.read_csv('events.csv', parse_dates=['start','end'])
coal_imports = pd.read_csv('coal_imports.csv')
fixtures = pd.read_csv('fixtures.csv', parse_dates=['Date'])
port_constraints = pd.read_csv('port_constraints.csv')
ports = pd.read_csv('ports.csv')

print("All 12 loaded:", freight.shape, ffa.shape, bunker.shape, coal_prices.shape,
      port_calls.shape, ais_positions.shape, weather.shape, events.shape,
      coal_imports.shape, fixtures.shape, port_constraints.shape, ports.shape)

def norm(series, low=None, high=None):
    low = series.min() if low is None else low
    high = series.max() if high is None else high
    if high == low: return series * 0 + 50
    return ((series - low) / (high - low) * 100).clip(0, 100)

All 12 loaded: (4320, 7) (17280, 5) (3240, 5) (360, 6) (1121, 7) (7861, 8) (1830, 11) (34, 7) (350, 5) (1512, 10) (6, 15) (6, 6)


In [2]:
freight_vol = freight.groupby('date')['freight_usd_mt'].mean().rolling(3, min_periods=1).std().fillna(0)
bunker_vol = bunker.groupby('date')['price'].mean().rolling(3, min_periods=1).std().fillna(0)
coal_vol = coal_prices.groupby('date')['price'].mean().rolling(3, min_periods=1).std().fillna(0)

market_risk_by_date = pd.DataFrame({
    'freight_vol': norm(freight_vol), 'bunker_vol': norm(bunker_vol), 'coal_vol': norm(coal_vol),
}).mean(axis=1)
print(market_risk_by_date.tail())

date
2025-07-03    19.997108
2025-08-02    24.756992
2025-09-01    19.777934
2025-10-01    26.400127
2025-10-31    10.052025
dtype: float64


In [3]:
port_wait_by_port = port_calls.groupby('port_id')['waiting_hours'].agg(['mean','min','max'])
port_calls['port_risk'] = port_calls.apply(
    lambda r: norm(pd.Series([r['waiting_hours']]),
                   low=port_wait_by_port.loc[r['port_id'],'min'],
                   high=port_wait_by_port.loc[r['port_id'],'max']).iloc[0], axis=1)
port_risk_hist = port_calls.groupby('port_id')['port_risk'].mean()

port_constraints_idx = port_constraints.set_index('port_id')
structural_risk = (norm(-port_constraints_idx['berth_count']) * 0.5 +
                    norm(-port_constraints_idx['max_draft']) * 0.5)

port_risk_by_port = (port_risk_hist * 0.6) + (structural_risk * 0.4)

# sanity check that ais_positions genuinely underlies this port_calls data
print("ais_positions vessels all exist in port_calls:",
      set(ais_positions['imo']).issubset(set(port_calls['imo'])))
print(port_risk_by_port)

ais_positions vessels all exist in port_calls: True
port_id
DHA    19.092263
GAN    46.533371
GOP    58.938876
HAL    63.694505
PAR    27.819660
VIZ    31.998838
dtype: float64


In [4]:
weather['weather_risk'] = (norm(weather['wind_speed'])*0.35 + norm(weather['wave_height'])*0.25 +
                            weather['storm_flag']*40 + weather['cyclone_flag']*60).clip(0,100)
weather_risk_by_port = weather.groupby('port_id')['weather_risk'].mean()
print(weather_risk_by_port)

port_id
DHA    16.912498
GAN    17.780659
GOP    17.276586
HAL    18.574925
PAR    16.250245
VIZ    17.655017
Name: weather_risk, dtype: float64


In [5]:
severity_map = {'Low': 25, 'Medium': 55, 'High': 85}

def geopolitical_risk(route_id, date):
    date = pd.Timestamp(date)
    active = events[(events['start'] <= date) & (events['end'] >= date) &
                     events['affected_routes'].apply(lambda s: route_id in str(s).split(';'))]
    return 10 if len(active) == 0 else active['severity'].map(severity_map).max()

print("Russia route during war:", geopolitical_risk('RUS_PAR_PAN', '2022-06-19'))
print("Unaffected route:", geopolitical_risk('AUS_PAR_PAN', '2020-01-01'))

Russia route during war: 85
Unaffected route: 10


In [6]:
coal_imports_sorted = coal_imports.sort_values(['origin_country','month'])
coal_imports_sorted['rolling_avg'] = coal_imports_sorted.groupby('origin_country')['quantity_mt']\
    .transform(lambda x: x.rolling(6, min_periods=1).mean())
coal_imports_sorted['pct_deviation'] = ((coal_imports_sorted['quantity_mt'] - coal_imports_sorted['rolling_avg'])
                                         / coal_imports_sorted['rolling_avg']).abs()*100
coal_imports_sorted['supply_risk'] = norm(coal_imports_sorted['pct_deviation'], low=0, high=50)
supply_risk_by_country = coal_imports_sorted.groupby('origin_country')['supply_risk'].mean()
print(supply_risk_by_country)

origin_country
Australia     25.637609
Indonesia     31.189386
Mozambique    28.750936
Russia        24.857379
USA           28.274290
Name: supply_risk, dtype: float64


In [7]:
fixtures['route_key'] = fixtures['Origin'] + "-" + fixtures['Destination']
spot_share = fixtures.groupby('route_key')['Contract type'].apply(lambda x: (x=='Spot').mean()*100)
print(spot_share.head())

route_key
Baltimore-Dhamra        57.142857
Baltimore-Gangavaram    76.190476
Baltimore-Gopalpur      53.846154
Baltimore-Haldia        29.411765
Baltimore-Paradip       62.500000
Name: Contract type, dtype: float64


In [8]:
port_names = ports.set_index('port_id')['name']
PORT_NAME = {"DHA":"Dhamra","GAN":"Gangavaram","GOP":"Gopalpur","HAL":"Haldia","PAR":"Paradip","VIZ":"Vizag"}
WEIGHTS = {"market":1/6, "port":1/6, "weather":1/6, "geopolitical":1/6, "supply":1/6, "contract":1/6}

def get_risk_score(route_id, origin_country, destination_port, date):
    date_ts = pd.Timestamp(date)
    destination_port = destination_port.upper().strip()
    nearest_date = market_risk_by_date.index[(market_risk_by_date.index - date_ts).to_series().abs().argmin()]
    market = float(market_risk_by_date.get(nearest_date, market_risk_by_date.mean()))
    port = float(port_risk_by_port.get(destination_port, port_risk_by_port.mean()))
    weather_s = float(weather_risk_by_port.get(destination_port, weather_risk_by_port.mean()))
    geo = geopolitical_risk(route_id, date_ts)
    supply = float(supply_risk_by_country.get(origin_country, supply_risk_by_country.mean()))
    route_key = f"{origin_country}-{PORT_NAME.get(destination_port, destination_port)}"
    contract = float(spot_share.get(route_key, spot_share.mean()))
    scores = {"market": market, "port": port, "weather": weather_s, "geopolitical": geo,
              "supply": supply, "contract": contract}
    overall = sum(scores[k]*WEIGHTS[k] for k in scores)
    return {"route_id": route_id, "destination_port_name": port_names.get(destination_port, destination_port),
            "date": date, **{k: round(v,1) for k,v in scores.items()}, "overall": round(overall,1)}

print(get_risk_score("RUS_PAR_PAN", "Russia", "PAR", "2022-06-19"))
print(get_risk_score("AUS_DHA_CAP", "Australia", "DHA", "2020-01-01"))
print(get_risk_score("USA_HAL_PAN", "USA", "HAL", "2023-07-15"))

{'route_id': 'RUS_PAR_PAN', 'destination_port_name': 'Paradip', 'date': '2022-06-19', 'market': 64.2, 'port': 27.8, 'weather': 16.3, 'geopolitical': np.int64(85), 'supply': 24.9, 'contract': 59.5, 'overall': np.float64(46.3)}
{'route_id': 'AUS_DHA_CAP', 'destination_port_name': 'Dhamra', 'date': '2020-01-01', 'market': 0.0, 'port': 19.1, 'weather': 16.9, 'geopolitical': 10, 'supply': 25.6, 'contract': 59.5, 'overall': 21.9}
{'route_id': 'USA_HAL_PAN', 'destination_port_name': 'Haldia', 'date': '2023-07-15', 'market': 20.3, 'port': 63.7, 'weather': 18.6, 'geopolitical': 10, 'supply': 28.3, 'contract': 59.5, 'overall': 33.4}


In [9]:
market_risk_by_date.reset_index().rename(columns={0:'market_risk'}).to_csv('market_risk_by_date.csv', index=False)
port_risk_by_port.reset_index().rename(columns={0:'port_risk'}).to_csv('port_risk_by_port.csv', index=False)
weather_risk_by_port.reset_index().rename(columns={'weather_risk':'weather_risk'}).to_csv('weather_risk_by_port.csv', index=False)
supply_risk_by_country.reset_index().rename(columns={'supply_risk':'supply_risk'}).to_csv('supply_risk_by_country.csv', index=False)
spot_share.reset_index().rename(columns={'Contract type':'contract_risk'}).to_csv('contract_risk_by_route.csv', index=False)
events.to_csv('events_lookup.csv', index=False)
ports[['port_id','name']].to_csv('port_names.csv', index=False)
print("Lookup tables saved.")

Lookup tables saved.
